In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### load the Dataset

In [2]:
df = pd.read_csv('hearing_test.csv')
df.head()

,age,physical_score,test_result
0,33.0,40.7,1
1,50.0,37.2,1
2,52.0,24.7,0
3,56.0,31.0,0
4,35.0,42.9,1


### EDA

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             5000 non-null   float64
 1   physical_score  5000 non-null   float64
 2   test_result     5000 non-null   int64  
dtypes: float64(2), int64(1)
memory usage: 117.3 KB


In [4]:
df.describe()

,age,physical_score,test_result
count,5000.000000,5000.000000,5000.000000
mean,51.609000,32.760260,0.600000
std,11.287001,8.169802,0.489947
min,18.000000,-0.000000,0.000000
25%,43.000000,26.700000,0.000000
50%,51.000000,35.300000,1.000000
75%,60.000000,38.900000,1.000000
max,90.000000,50.000000,1.000000


In [5]:
df.isna().sum()

age               0
physical_score    0
test_result       0
dtype: int64

In [6]:
df.corr()

,age,physical_score,test_result
age,1.000000,-0.782146,-0.683171
physical_score,-0.782146,1.000000,0.792716
test_result,-0.683171,0.792716,1.000000


In [17]:
#check the dataset is imbalanced or not 
df['test_result'].value_counts()

test_result
1    3000
0    2000
Name: count, dtype: int64

In [18]:
df_majority = df[df['test_result'] == 1]
df_minority = df[df['test_result'] == 0]

In [34]:
from sklearn.utils import resample

df_majority_under = resample(
    df_majority,
    replace=False,
    n_samples=2500,
    random_state=42
)
df_majority_under.shape

(2500, 3)

In [35]:
df_balanced = pd.concat([df_majority_under, df_minority])
df_balanced.shape

(4500, 3)

### DATA Prepration

In [36]:
#split the data
x = df_balanced.drop('test_result',axis=1)
y = df_balanced['test_result']

In [37]:
from imblearn.over_sampling import SMOTE
smote=SMOTE(random_state=42)
x_new,y_new=smote.fit_resample(x,y)

In [38]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x_new,y_new,train_size=0.7,random_state=42)

### Build the Model

In [39]:
from sklearn.linear_model import LogisticRegressionCV
def build_logistic_model():
    model = LogisticRegressionCV()
    model.fit(x_train,y_train)
    return model

In [42]:
from sklearn.naive_bayes import GaussianNB
def build_naive_bayes_model():
    model = GaussianNB()
    model.fit(x_train,y_train)
    return model

In [49]:
from sklearn.neighbors import KNeighborsClassifier
def build_knn_model():
    model = KNeighborsClassifier(n_neighbors=70)
    model.fit(x_train,y_train)
    return model

### evaluate the Model

In [50]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,roc_curve,log_loss
def evaluate_model(model_name,model,x,y):
    #predict x
    y_pred = model.predict(x)
    accuracy = accuracy_score(y,y_pred)
    presision = precision_score(y,y_pred)
    recall = recall_score(y,y_pred)
    f1 = f1_score(y,y_pred)
    loss= log_loss(y,y_pred)
    return accuracy,presision,recall,f1,loss

In [51]:
#create the all models

In [52]:
model_logistic_regression = build_logistic_model()
model_naive_bayes_model = build_naive_bayes_model()
model_knn = build_knn_model()
models = [
    {'name':'Logistic Regrresion','model':model_logistic_regression},
    {'name':'naive bayes','model':model_naive_bayes_model},
    {'name':'KNN Model','model':model_knn}
]

C:\Users\Hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
C:\Users\Hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:2137: FutureWarning: The default value of the parameter 'scoring' will change from None, i.e. accuracy, to 'neg_log_loss' in version 1.11. To silence this warning, explicitly set the scoring parameter: scoring='neg_log_loss' for the new, scoring='accuracy' or scoring=None for the old default.
  warnings.warn(
C:\Users\Hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:2150: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplifie

### Evaluation metrics for training data

In [53]:
evaluation_metrics=[]
for model_info in models:
    evaluation_metrics.append(evaluate_model(model_info['name'],model_info['model'],x_train,y_train))
pd.DataFrame(evaluation_metrics,columns=['Accuracy','Presion','Recall','F1','Loss'])

,Accuracy,Presion,Recall,F1,Loss
0,0.911429,0.899438,0.924365,0.911731,3.192438
1,0.909714,0.899549,0.920323,0.909817,3.254227
2,0.920286,0.897212,0.947460,0.921651,2.873194


### Evaluation for Test Data

In [54]:
evaluation_metrics=[]
for model_info in models:
    evaluation_metrics.append(evaluate_model(model_info['name'],model_info['model'],x_test,y_test))
pd.DataFrame(evaluation_metrics,columns=['Accuracy','Presion','Recall','F1','Loss'])

,Accuracy,Presion,Recall,F1,Loss
0,0.899333,0.889029,0.917969,0.903267,3.628394
1,0.894667,0.892031,0.903646,0.897801,3.796598
2,0.906667,0.886700,0.937500,0.911392,3.364074
